# Traffic Light Detector — Approach 2: EfficientNet-B3 + Advanced Training

**Task**: Binary classification — *is a traffic light visible in the frame?*  
**Dataset**: CARLA driving simulator (front-facing RGB camera)  
**Backbone**: EfficientNet-B3 pretrained on ImageNet (via `timm`)  
**Key techniques**: Focal Loss · WeightedRandomSampler · MixUp + RandAugment  
  AdamW + OneCycleLR · Progressive unfreezing · GradCAM · TTA  
**Environment**: Google Colab Pro · A100 GPU (40 GB)  

---

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
print(f'PyTorch: {torch.__version__}  |  CUDA: {torch.version.cuda}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  |  VRAM: {props.total_memory / 1e9:.1f} GB')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
torch.set_float32_matmul_precision('high')

In [ ]:
!pip install -q timm scikit-learn matplotlib seaborn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, json, random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.cm as cm_plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from torchvision.transforms import RandAugment
import timm

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report,
    roc_curve, precision_score, recall_score,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

In [ ]:
TARGET    = 'has_traffic_light'
TASK_NAME = 'TrafficLight'

DRIVE_ROOT = Path('/content/drive/MyDrive/ML_Safety_2026')
LOCAL      = Path('/content/ML_Safety_2026')   # local SSD mirror

TRAIN_DIR  = LOCAL / 'train'      / 'train'
VAL_DIR    = LOCAL / 'validation' / 'validation'
TEST_DIR   = LOCAL / 'test'       / 'test'
CKPT_DIR   = DRIVE_ROOT / 'checkpoints' / TASK_NAME / 'EfficientNetB3'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE     = 300
BATCH_SIZE   = 128
NUM_WORKERS  = 4
NUM_EPOCHS   = 10
LR_MAX       = 5e-4
WEIGHT_DECAY = 1e-2
FOCAL_GAMMA  = 2.0
FOCAL_ALPHA  = 0.25
MIXUP_ALPHA  = 0.4
THRESHOLD    = 0.5
N_TTA        = 6
# Adapted for 10 epochs: epochs 1-3 head-only, 4-7 partial, 8-10 full unfreeze
UNFREEZE_SCHEDULE = {1: 0, 4: 4, 8: 7}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}  |  Task: {TASK_NAME}  ({TARGET})')

In [ ]:
# ── Copy dataset to local SSD (one-time, ~5-10 min → epochs 3-5x faster) ─────
import shutil
if not LOCAL.exists():
    print(f'Copying {DRIVE_ROOT} → {LOCAL} ...')
    shutil.copytree(str(DRIVE_ROOT), str(LOCAL))
    size_gb = sum(f.stat().st_size for f in LOCAL.rglob('*') if f.is_file()) / 1e9
    print(f'Done — {size_gb:.2f} GB on local SSD.')
else:
    print(f'Local copy already present at {LOCAL}')

In [ ]:
class CarlaDataset(Dataset):
    def __init__(self, split_dir, target_col, transform=None):
        split_dir = Path(split_dir)
        df = pd.read_csv(split_dir / 'labels.csv')
        df['frame'] = df['frame'].astype(str).str.zfill(6)
        df['img_path'] = df['frame'].apply(
            lambda f: str(split_dir / 'rgb-front' / f'{f}.jpg')
        )
        exists = df['img_path'].apply(os.path.exists)
        df = df[exists].reset_index(drop=True)
        df[target_col] = df[target_col].map(
            {True: 1, False: 0, 'True': 1, 'False': 0}
        ).astype(int)
        self.df = df; self.target_col = target_col; self.transform = transform

    def __len__(self):  return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, torch.tensor(float(row[self.target_col]), dtype=torch.float32)

    def make_weighted_sampler(self):
        labels  = self.df[self.target_col].values
        counts  = np.bincount(labels)
        weights = 1.0 / counts[labels]
        print(f'  Negative: {counts[0]:,}  Positive: {counts[1]:,}  ratio {counts[0]/max(counts[1],1):.2f}:1')
        return WeightedRandomSampler(
            weights=torch.DoubleTensor(weights),
            num_samples=len(labels),
            replacement=True,
        )

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE + 40, IMG_SIZE + 40)),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
tta_tfs = [
    transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN,STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(MEAN,STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE+20,IMG_SIZE+20)), transforms.CenterCrop(IMG_SIZE), transforms.ToTensor(), transforms.Normalize(MEAN,STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE+20,IMG_SIZE+20)), transforms.CenterCrop(IMG_SIZE), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize(MEAN,STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.ColorJitter(brightness=0.1,contrast=0.1), transforms.ToTensor(), transforms.Normalize(MEAN,STD)]),
    transforms.Compose([transforms.Resize((IMG_SIZE,IMG_SIZE)), transforms.RandomRotation(5), transforms.ToTensor(), transforms.Normalize(MEAN,STD)]),
]

print('Building datasets ...')
train_ds = CarlaDataset(TRAIN_DIR, TARGET, train_tf)
val_ds   = CarlaDataset(VAL_DIR,   TARGET, val_tf)
test_ds  = CarlaDataset(TEST_DIR,  TARGET, val_tf)
print('Train:'); sampler = train_ds.make_weighted_sampler()
print(f'  {len(train_ds):,}  Val: {len(val_ds):,}  Test: {len(test_ds):,}')

kw = dict(num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **kw)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **kw)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **kw)

In [ ]:
def denorm(t):
    t = t.permute(1, 2, 0).numpy()
    return np.clip(t * np.array(STD) + np.array(MEAN), 0, 1)

vis_ds = CarlaDataset(TRAIN_DIR, TARGET, val_tf)
idxs   = random.sample(range(len(vis_ds)), 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, idx in enumerate(idxs):
    img, lbl = vis_ds[idx]
    ax = axes[i // 4][i % 4]
    ax.imshow(denorm(img))
    color = 'limegreen' if lbl.item() == 1 else 'tomato'
    ax.set_title('Present' if lbl.item() == 1 else 'Absent', color=color, fontweight='bold', fontsize=11)
    ax.axis('off')
plt.suptitle(f'Sample frames — {TASK_NAME}', fontsize=14)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'samples.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, alpha=0.25, reduction='mean'):
        super().__init__()
        self.gamma = gamma; self.alpha = alpha; self.reduction = reduction

    def forward(self, logits, targets):
        bce     = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t     = torch.exp(-bce)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        loss    = alpha_t * (1 - p_t) ** self.gamma * bce
        return loss.mean() if self.reduction == 'mean' else loss.sum()

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA)

def mixup_batch(imgs, labels, alpha=0.4):
    if alpha <= 0: return imgs, labels
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(imgs.size(0), device=imgs.device)
    return lam * imgs + (1-lam) * imgs[idx], lam * labels + (1-lam) * labels[idx]

In [ ]:
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=1,
                          drop_rate=0.3, drop_path_rate=0.2).to(DEVICE)
n_total = sum(p.numel() for p in model.parameters())
print(f'EfficientNet-B3 — {n_total/1e6:.2f}M parameters')

def set_frozen_stages(model, n_stages_unfrozen):
    for p in model.parameters(): p.requires_grad = False
    for p in model.classifier.parameters(): p.requires_grad = True
    if n_stages_unfrozen > 0 and hasattr(model, 'blocks'):
        for layer in [model.conv_head, model.bn2]:
            for p in layer.parameters(): p.requires_grad = True
        stages = list(model.blocks.children())
        for stage in stages[max(0, len(stages) - n_stages_unfrozen):]:
            for p in stage.parameters(): p.requires_grad = True
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'  Trainable: {n_tr/1e6:.2f}M / {n_total/1e6:.2f}M  [{n_stages_unfrozen} stages unfrozen]')

set_frozen_stages(model, UNFREEZE_SCHEDULE[1])

In [ ]:
def make_optimizer(model, lr=LR_MAX):
    return optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=lr, weight_decay=WEIGHT_DECAY)

optimizer = make_optimizer(model)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR_MAX, steps_per_epoch=len(train_loader),
    epochs=NUM_EPOCHS, pct_start=0.1, div_factor=25, final_div_factor=1e4,
)
# No GradScaler needed: A100 + bfloat16 has sufficient dynamic range
print('AdamW + OneCycleLR configured')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    tot_loss, preds_all, labels_all = 0.0, [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        imgs, labels = mixup_batch(imgs, labels, MIXUP_ALPHA)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(imgs).squeeze(1); loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()  # OneCycleLR steps per batch
        tot_loss += loss.item() * len(labels)
        hard = (labels > 0.5).long().cpu().numpy()
        p = (torch.sigmoid(logits) > THRESHOLD).long().cpu().numpy()
        preds_all.extend(p); labels_all.extend(hard)
    return tot_loss / len(loader.dataset), f1_score(labels_all, preds_all, zero_division=0)

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_loss, probs_all, preds_all, labels_all = 0.0, [], [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE, non_blocking=True), labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
            logits = model(imgs).squeeze(1); loss = criterion(logits, labels)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        tot_loss += loss.item() * len(labels)
        probs_all.extend(probs); preds_all.extend((probs>THRESHOLD).astype(int))
        labels_all.extend(labels.long().cpu().numpy())
    la = np.array(labels_all)
    return dict(
        loss=tot_loss/len(loader.dataset), acc=accuracy_score(la,preds_all),
        f1=f1_score(la,preds_all,zero_division=0),
        auc=roc_auc_score(la,probs_all) if len(np.unique(la))>1 else 0.5,
        prec=precision_score(la,preds_all,zero_division=0),
        rec=recall_score(la,preds_all,zero_division=0),
        probs=probs_all, labels=labels_all,
    )

In [ ]:
history  = {k: [] for k in ('tr_loss','tr_f1','val_loss','val_f1','val_auc','lr')}
best_f1  = 0.0; best_auc = 0.0

print('=' * 65)
print(f'Training EfficientNet-B3 — {TASK_NAME} ({NUM_EPOCHS} epochs)')
print('Progressive unfreeze schedule:', UNFREEZE_SCHEDULE)
print('=' * 65)

for ep in range(1, NUM_EPOCHS + 1):
    if ep in UNFREEZE_SCHEDULE:
        n_stages = UNFREEZE_SCHEDULE[ep]
        print(f'\n[Ep {ep}] Unfreezing {n_stages} block stages ...')
        set_frozen_stages(model, n_stages)
        # Extend existing optimizer with newly unfrozen params — keeps scheduler linked
        existing_ids = {id(p) for group in optimizer.param_groups for p in group['params']}
        new_params = [p for p in model.parameters()
                      if p.requires_grad and id(p) not in existing_ids]
        if new_params:
            optimizer.add_param_group({
                'params': new_params,
                'lr': optimizer.param_groups[0]['lr'],
                'weight_decay': WEIGHT_DECAY,
            })

    tl, tf = train_epoch(model, train_loader, optimizer, scheduler)
    vm     = evaluate(model, val_loader)
    lr_now = optimizer.param_groups[0]['lr']

    history['tr_loss'].append(tl); history['tr_f1'].append(tf)
    history['val_loss'].append(vm['loss']); history['val_f1'].append(vm['f1'])
    history['val_auc'].append(vm['auc']); history['lr'].append(lr_now)

    mark = ''
    if vm['f1'] >= best_f1:
        best_f1 = vm['f1']; best_auc = vm['auc']
        torch.save(model.state_dict(), CKPT_DIR / 'best_model.pth'); mark = '  ✓'
    print(f'Ep {ep:02d}/{NUM_EPOCHS}  lr={lr_now:.2e}  '
          f'tr {tl:.4f}/{tf:.4f}  val {vm["loss"]:.4f}/{vm["f1"]:.4f}/{vm["auc"]:.4f}{mark}')

print(f'\nBest → F1 {best_f1:.4f}  AUC {best_auc:.4f}')

In [ ]:
ep = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
milestone_eps = sorted(UNFREEZE_SCHEDULE.keys())
for ax in axes.flat:
    for me in milestone_eps[1:]: ax.axvline(me-0.5, ls=':', color='gray', alpha=0.4)

axes[0,0].plot(ep, history['tr_loss'], lw=2, label='Train')
axes[0,0].plot(ep, history['val_loss'], lw=2, label='Val')
axes[0,0].set(title='Focal Loss', xlabel='Epoch'); axes[0,0].legend()

axes[0,1].plot(ep, history['tr_f1'], lw=2, label='Train')
axes[0,1].plot(ep, history['val_f1'], lw=2, label='Val')
axes[0,1].set(title='F1 Score', xlabel='Epoch', ylim=[0,1]); axes[0,1].legend()

axes[1,0].plot(ep, history['val_auc'], color='darkorange', lw=2)
axes[1,0].set(title='Val AUC-ROC', xlabel='Epoch', ylim=[0,1])

axes[1,1].semilogy(ep, history['lr'], color='purple', lw=2)
axes[1,1].set(title='LR (OneCycle)', xlabel='Epoch')

plt.suptitle(f'Training Curves — {TASK_NAME} (EfficientNet-B3)', fontsize=14)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / 'best_model.pth', map_location=DEVICE))
model.eval()

class TTADataset(Dataset):
    def __init__(self, base_ds, tf):
        self.base_ds = base_ds; self.tf = tf
    def __len__(self): return len(self.base_ds)
    def __getitem__(self, idx):
        row = self.base_ds.df.iloc[idx]
        img = Image.open(row['img_path']).convert('RGB')
        return self.tf(img), torch.tensor(float(row[self.base_ds.target_col]), dtype=torch.float32)

@torch.no_grad()
def tta_predict(base_ds, tta_transforms):
    all_probs_list = []; all_labels = None
    kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=True)
    for i, tf in enumerate(tta_transforms):
        ds = TTADataset(base_ds, tf)
        loader = DataLoader(ds, shuffle=False, **kw)
        probs_i, labels_i = [], []
        for imgs, labels in loader:
            imgs = imgs.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
                logits = model(imgs).squeeze(1)
            probs_i.extend(torch.sigmoid(logits).float().cpu().numpy())
            labels_i.extend(labels.numpy())
        all_probs_list.append(probs_i)
        if all_labels is None: all_labels = labels_i
        print(f'  TTA pass {i+1}/{len(tta_transforms)} done')
    return np.mean(all_probs_list, axis=0), np.array(all_labels)

print('Running TTA ...')
tta_probs, tta_labels = tta_predict(test_ds, tta_tfs[:N_TTA])
tta_preds = (tta_probs > THRESHOLD).astype(int)
tta_auc   = roc_auc_score(tta_labels, tta_probs) if len(np.unique(tta_labels))>1 else 0.5

print(f'\n=== TTA Test Results — {TASK_NAME} (EfficientNet-B3) ===')
print(f'F1: {f1_score(tta_labels,tta_preds,zero_division=0):.4f}  AUC: {tta_auc:.4f}')
print(classification_report(tta_labels, tta_preds, target_names=['Absent','Present']))

In [ ]:
tm = evaluate(model, test_loader)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
cm = confusion_matrix(tta_labels, tta_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['Absent','Present'], yticklabels=['Absent','Present'])
axes[0].set(title=f'Confusion Matrix (TTA) — {TASK_NAME}', xlabel='Predicted', ylabel='True')
fpr_b, tpr_b, _ = roc_curve(tm['labels'], tm['probs'])
fpr_t, tpr_t, _ = roc_curve(tta_labels, tta_probs)
axes[1].plot(fpr_b, tpr_b, lw=2, label=f'No TTA  AUC={tm["auc"]:.4f}')
axes[1].plot(fpr_t, tpr_t, lw=2, label=f'TTA  AUC={tta_auc:.4f}')
axes[1].plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
axes[1].set(title='ROC Curve', xlabel='FPR', ylabel='TPR', xlim=[0,1], ylim=[0,1])
axes[1].legend(fontsize=10)
plt.suptitle(f'Test Evaluation — {TASK_NAME} (EfficientNet-B3)', fontsize=14)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'test_results.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── GradCAM ──────────────────────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model; self.grads = None; self.acts = None
        target_layer.register_forward_hook(lambda m,i,o: setattr(self,'acts',o.detach()))
        target_layer.register_full_backward_hook(lambda m,i,o: setattr(self,'grads',o[0].detach()))

    def __call__(self, img_tensor):
        self.model.zero_grad()
        logit = self.model(img_tensor.unsqueeze(0).to(DEVICE)).squeeze()
        logit.backward()
        cam = (self.grads.mean(dim=(2,3), keepdim=True) * self.acts).sum(dim=1).squeeze()
        cam = F.relu(cam); cam = cam - cam.min(); cam = cam / (cam.max() + 1e-8)
        return cam.cpu().numpy()

grad_cam = GradCAM(model, model.conv_head)
vis_ds   = CarlaDataset(TEST_DIR, TARGET, val_tf)
model.train()
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
idxs = random.sample(range(len(vis_ds)), 4)
for i, idx in enumerate(idxs):
    img_t, lbl = vis_ds[idx]
    cam   = grad_cam(img_t)
    img_np = np.clip(img_t.permute(1,2,0).numpy() * np.array(STD) + np.array(MEAN), 0, 1)
    axes[0,i].imshow(img_np)
    axes[0,i].set_title(f'GT: {"Present" if lbl==1 else "Absent"}', fontsize=10); axes[0,i].axis('off')
    cam_r = np.array(Image.fromarray((cam*255).astype(np.uint8)).resize(
        (img_np.shape[1], img_np.shape[0]), Image.BILINEAR)) / 255.0
    overlay = np.clip(0.6*img_np + 0.4*cm_plt.jet(cam_r)[:,:,:3], 0, 1)
    axes[1,i].imshow(overlay)
    axes[1,i].set_title('GradCAM', fontsize=10); axes[1,i].axis('off')
plt.suptitle(f'GradCAM — {TASK_NAME} (Test set)', fontsize=13)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'gradcam.png', dpi=120, bbox_inches='tight')
plt.show()
model.eval()

In [ ]:
results = {
    'approach': 'Approach 2 — EfficientNet-B3 + Advanced Training',
    'target': TARGET, 'task': TASK_NAME,
    'hparams': dict(img_size=IMG_SIZE, batch_size=BATCH_SIZE, num_epochs=NUM_EPOCHS,
                    lr_max=LR_MAX, weight_decay=WEIGHT_DECAY,
                    focal_gamma=FOCAL_GAMMA, focal_alpha=FOCAL_ALPHA,
                    mixup_alpha=MIXUP_ALPHA, n_tta=N_TTA),
    'test_no_tta': dict(accuracy=round(tm['acc'],4), f1=round(tm['f1'],4),
                        precision=round(tm['prec'],4), recall=round(tm['rec'],4), auc=round(tm['auc'],4)),
    'test_tta': dict(
        accuracy=round(float(accuracy_score(tta_labels,tta_preds)),4),
        f1=round(float(f1_score(tta_labels,tta_preds,zero_division=0)),4),
        precision=round(float(precision_score(tta_labels,tta_preds,zero_division=0)),4),
        recall=round(float(recall_score(tta_labels,tta_preds,zero_division=0)),4),
        auc=round(float(tta_auc),4),
    ),
    'val_best_f1': round(best_f1,4), 'val_best_auc': round(best_auc,4),
}
out = CKPT_DIR / 'results.json'
with open(out, 'w') as f: json.dump(results, f, indent=2)
print(f'Saved → {out}')
print(json.dumps(results, indent=2))